# 1. Introducción al Aprendizaje Automático

Temas selectos de aprendizaje de máquina \
Ana Luisa Llamas Martinez

## 1.3. Generación de los datos

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 300

superficie = rng.normal(90, 25, size=n).clip(30, None)          # m^2
habitaciones = rng.integers(1, 6, size=n).astype(float)
ciudad = rng.choice(["Xalapa", "CDMX", "Monterrey"], size=n,
                     p=[0.5, 0.3, 0.2])
antiguedad = rng.integers(0, 40, size=n).astype(float)           # años

# Precio "verdadero" simulado: relacion lineal + ruido aleatorio
bonus_ciudad = pd.Series(ciudad).map(
    {"Xalapa": 0, "CDMX": 450, "Monterrey": 200}).to_numpy()

precio = (
    15 * superficie
    + 80 * habitaciones
    - 3 * antiguedad
    + bonus_ciudad
    + rng.normal(0, 150, size=n)
)

df = pd.DataFrame({
    "superficie": superficie,
    "habitaciones": habitaciones,
    "ciudad": ciudad,
    "antiguedad": antiguedad,
    "precio": precio,
})

# Introducimos valores faltantes A PROPOSITO8
faltantes_idx = rng.choice(df.index, size=20, replace=False)
df.loc[faltantes_idx, "habitaciones"] = np.nan

df.head()

,superficie,habitaciones,ciudad,antiguedad,precio
0,97.617927,NaN,Monterrey,24.0,1946.102881
1,64.000397,2.0,Xalapa,1.0,1259.153402
2,108.761280,NaN,CDMX,12.0,2038.753463
3,113.514118,2.0,CDMX,5.0,2121.916030
4,41.224120,3.0,CDMX,0.0,1144.363226


### 1.4. Análisis exploratorio (EDA)

In [2]:
#tipos de datos y cuántos valores no nulos hay en cada columna

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   superficie    300 non-null    float64
 1   habitaciones  280 non-null    float64
 2   ciudad        300 non-null    object 
 3   antiguedad    300 non-null    float64
 4   precio        300 non-null    float64
dtypes: float64(4), object(1)
memory usage: 11.8+ KB


In [3]:
#estadísticos básicos de las columnas numéricas

df.describe()

,superficie,habitaciones,antiguedad,precio
count,300.000000,280.000000,300.000000,300.000000
mean,88.986916,3.032143,18.956667,1692.580762
std,23.220181,1.455075,11.776974,445.229373
min,30.000000,1.000000,0.000000,590.322203
25%,73.265883,2.000000,9.000000,1392.680281
50%,87.825240,3.000000,19.000000,1703.704045
75%,101.842244,4.000000,29.000000,1966.741023
max,162.846562,5.000000,39.000000,3319.653720


In [4]:
#cuántas viviendas hay de cada ciudad

df["ciudad"].value_counts()

ciudad
Xalapa       155
CDMX          89
Monterrey     56
Name: count, dtype: int64

## 1.5 Partición de datos: entrenamiento y prueba

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["precio"])
y = df["precio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

In [6]:
X_train.shape, X_test.shape

((240, 4), (60, 4))

## 1.6 Preprocesamiento: imputación, codificación y escalamiento

In [7]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

columnas_numericas = ["superficie", "habitaciones", "antiguedad"]
columnas_categoricas = ["ciudad"]

In [8]:
# Pipeline para columnas numéricas: imputar mediana + escalar
pipeline_numerico = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ('scaler', StandardScaler())
])

pipeline_numerico

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputar', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

In [9]:
pipeline_categorico = Pipeline([
    ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore")),
])

In [10]:
preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_numerico, columnas_numericas),
    ("cat", pipeline_categorico, columnas_categoricas),
])

## 1.7 Entrenamiento del modelo

In [11]:
from sklearn.linear_model import Ridge

modelo = Pipeline([
    ("preprocesamiento", preprocesador),
    ("regresor", Ridge(alpha=1.0))
])


In [12]:
modelo.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesamiento', ...), ('regresor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['superficie','habitaciones','ciudad','antiguedad']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated

## 1.8 Evaluación del modelo

In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TODO: obtén las predicciones sobre X_test
y_pred = modelo.predict(X_test)

# TODO: calcula RMSE, MAE y R^2
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
MAE = mean_absolute_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)

In [14]:
print(RMSE, MAE, R2)

145.4526209165449 111.47597440198575 0.9017987448744509


## 1.9 (Opcional) Validación cruzada

In [15]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=0)
puntajes = cross_val_score(
    modelo, X_train, y_train, cv=kf,
    scoring="neg_root_mean_squared_error")

print(f"RMSE promedio (CV, 5-fold): {-puntajes.mean():.2f}")

RMSE promedio (CV, 5-fold): 154.83


## 1.10 Bonus: clasificación con clases desbalanceadas

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix)

umbral = df["precio"].quantile(0.85)
y_clase = (df["precio"] >= umbral).astype(int)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_clase, test_size=0.2, random_state=0, stratify=y_clase)

clasificador = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(class_weight="balanced")),
])
clasificador.fit(Xc_train, yc_train)
yc_pred = clasificador.predict(Xc_test)

print("Matriz de confusión:\n", confusion_matrix(yc_test, yc_pred))
print("Exactitud :", round(accuracy_score(yc_test, yc_pred), 3))
print("Precisión :", round(precision_score(yc_test, yc_pred), 3))
print("Recall    :", round(recall_score(yc_test, yc_pred), 3))
print("F1        :", round(f1_score(yc_test, yc_pred), 3))

Matriz de confusión:
 [[45  6]
 [ 1  8]]
Exactitud : 0.883
Precisión : 0.571
Recall    : 0.889
F1        : 0.696


## 1.11. Preguntas de reflexión
Responde brevemente, en tus propias palabras:

1) ¿El modelo de regresión tuvo un desempeño razonable? ¿Cómo lo justificas con las métricas obtenidas?\
    **R:** Sí. RMSE fue de 145.45, que considerando el contexto de los precios de terrenos, no suena demasiado; RMA fue de 111.47, entonces la idea es similar. En cambio, el R2 fue de 0.9, entonces explica bastante bien la variabilidad.

2) ¿Qué habría pasado si el StandardScaler se hubiera ajustado con todos los datos antes de partir en entrenamiento y prueba? Relaciónalo con el concepto de data leakage.\
    **R:** Se hubiesen perdido datos, dado que se modifican los datos de entrenamiento, por ende, se podría tener un escenario demasiado positivo.

3) En la parte de clasificación, ¿por qué la exactitud (0.85–0.88 aprox.) no cuenta toda la historia? ¿Qué te dice la precisión que no te dice la exactitud?\
    **R:** Porque la exactitud es muy sensible a casos donde una clase predomina, en cambio, la precisión se enfoca en los falsos positivos.

4) Si quisieras predecir si una vivienda se vende en menos de 30 días en vez de su precio, ¿qué partes del pipeline cambiarías y cuáles dejarías igual?\
    **R:** La vriable respuesta cambiaría a ser una categórica: "sí se vende" o "no se vende". El resto, se mantiene.
